In [ ]:
# ── Cài Đặt Thư Viện Đọc PLY & Thư Viện Biểu Diễn 3D Trực Quan Plotly ─────────
!pip install -q plyfile plotly numpy pandas

In [ ]:
# ── KHÂU 1: Hiển Thị Mây Điểm Ban Đầu (Iter 0) Tương Tác 3D Nền Đen & Quỹ Đạo Camera ─────
import os
import json
import numpy as np
from plyfile import PlyData
import plotly.graph_objects as go

def display_interactive_point_cloud_plotly(ply_path, poses_path=None, fixed_cams_path=None, max_points=50000):
    """
    Hiển thị Mây Điểm 3D Tương Tác Nền Đen Tuyền (Black Background).
    - Hỗ trợ xoay tự do 360° mọi hướng (dragmode='orbit').
    - Đọc poses_bounds.npy để hiển thị 10 vị trí & quỹ đạo Camera thực tế.
    - Nút '▶ Tự Động Quay 360°' lướt qua quỹ đạo camera chuẩn xác.
    """
    if not os.path.exists(ply_path):
        print(f"❌ Không tìm thấy file point cloud tại: {ply_path}")
        return

    print(f"📂 Đang nạp Mây Điểm 3D Khâu 1: {ply_path} ...")
    plydata = PlyData.read(ply_path)
    vertex = plydata['vertex']

    x = np.asarray(vertex['x'], dtype=np.float32)
    y = np.asarray(vertex['y'], dtype=np.float32)
    z = np.asarray(vertex['z'], dtype=np.float32)

    names = [p.name for p in vertex.properties]
    if 'red' in names and 'green' in names and 'blue' in names:
        r = np.asarray(vertex['red'], dtype=np.float32)
        g = np.asarray(vertex['green'], dtype=np.float32)
        b = np.asarray(vertex['blue'], dtype=np.float32)
        if r.max() > 1.0:
            r, g, b = r / 255.0, g / 255.0, b / 255.0
    elif 'f_dc_0' in names and 'f_dc_1' in names and 'f_dc_2' in names:
        f_dc_0 = np.asarray(vertex['f_dc_0'], dtype=np.float32)
        f_dc_1 = np.asarray(vertex['f_dc_1'], dtype=np.float32)
        f_dc_2 = np.asarray(vertex['f_dc_2'], dtype=np.float32)
        SH_C0 = 0.28209479177387814
        r = np.clip(0.5 + SH_C0 * f_dc_0, 0.0, 1.0)
        g = np.clip(0.5 + SH_C0 * f_dc_1, 0.0, 1.0)
        b = np.clip(0.5 + SH_C0 * f_dc_2, 0.0, 1.0)
    else:
        r = g = b = np.ones_like(x) * 0.8

    total_points = len(x)
    if total_points > max_points:
        idx = np.random.choice(total_points, max_points, replace=False)
        x, y, z = x[idx], y[idx], z[idx]
        r, g, b = r[idx], g[idx], b[idx]

    colors_hex = [f'rgb({int(ri*255)},{int(gi*255)},{int(bi*255)})' for ri, gi, bi in zip(r, g, b)]

    # 1. Thêm Mây Điểm 3D
    data_traces = [
        go.Scatter3d(
            x=x, y=y, z=z,
            mode='markers',
            marker=dict(size=2.2, color=colors_hex, opacity=0.92),
            hoverinfo='none',
            name='Mây Điểm 3D'
        )
    ]

    # 2. Xử lý poses_bounds.npy nếu có để vẽ Quỹ đạo & Vị trí Camera 3D thực tế
    if poses_path and os.path.exists(poses_path):
        try:
            poses_bounds = np.load(poses_path)
            poses = poses_bounds[:, :15].reshape(-1, 3, 5)
            all_cam_centers = poses[:, :3, 3]
            c_x = all_cam_centers[:, 1]
            c_y = -all_cam_centers[:, 0]
            c_z = all_cam_centers[:, 2]
            all_cam_centers = np.stack([c_x, c_y, c_z], axis=-1)

            selected_indices = list(range(0, len(all_cam_centers), len(all_cam_centers)//10))
            if fixed_cams_path and os.path.exists(fixed_cams_path):
                with open(fixed_cams_path, "r", encoding="utf-8") as f:
                    cam_meta = json.load(f)
                    selected_indices = cam_meta.get("selected_indices", selected_indices)

            cam_centers = all_cam_centers[selected_indices]
            
            data_traces.append(
                go.Scatter3d(
                    x=all_cam_centers[:, 0], y=all_cam_centers[:, 1], z=all_cam_centers[:, 2],
                    mode='lines',
                    line=dict(color='#38bdf8', width=3, dash='dash'),
                    name='Quỹ Đạo Camera'
                )
            )

            data_traces.append(
                go.Scatter3d(
                    x=cam_centers[:, 0], y=cam_centers[:, 1], z=cam_centers[:, 2],
                    mode='markers+text',
                    marker=dict(size=6, color='#f43f5e', symbol='diamond'),
                    text=[f"Cam {i+1}" for i in range(len(cam_centers))],
                    textposition="top center",
                    name='10 Camera Cố Định'
                )
            )
            print(f"✅ Đã nạp thông số {len(all_cam_centers)} Camera từ poses_bounds.npy!")
        except Exception as e:
            print(f"⚠️ Lưu ý khi nạp poses_bounds.npy: {e}")

    fig = go.Figure(data=data_traces)

    # 3. Tạo 60 khung hình animation xoay 360 độ quanh tâm mây điểm
    frames = []
    n_frames = 60
    r_cam = 2.4
    for i in range(n_frames):
        theta = 2 * np.pi * i / n_frames
        cam_x = r_cam * np.cos(theta)
        cam_y = r_cam * np.sin(theta)
        frames.append(go.Frame(
            layout=dict(
                scene_camera=dict(
                    eye=dict(x=cam_x, y=cam_y, z=0.9),
                    center=dict(x=0, y=0, z=0),
                    up=dict(x=0, y=0, z=1)
                )
            ),
            name=f"frame_{i}"
        ))
    fig.frames = frames

    # 4. Thiết lập Nền Đen Tuyền (#000000) & Dragmode Orbit xoay 360° tự do mọi hướng
    fig.update_layout(
        title=dict(
            text=f"🌐 <b>Khâu 1: Mây Điểm Ban Đầu (Iter 0)</b> | Số điểm: <b>{total_points:,}</b>",
            x=0.02, y=0.97,
            font=dict(size=16, color='#60a5fa')
        ),
        scene=dict(
            dragmode='orbit', # Cho phép xoay 360° tự do không bị khóa trục
            xaxis=dict(visible=False),
            yaxis=dict(visible=False),
            zaxis=dict(visible=False),
            aspectmode='data',
            bgcolor='#000000' # Nền Đen Tuyền
        ),
        paper_bgcolor='#000000', # Nền Đen Tuyền
        font=dict(color='#e2e8f0', family='system-ui'),
        margin=dict(l=0, r=0, b=0, t=40),
        height=700,
        showlegend=True,
        legend=dict(x=0.80, y=0.95, bgcolor='rgba(0,0,0,0.7)', font=dict(color='#ffffff')),
        updatemenus=[
            dict(
                type="buttons",
                showactive=True,
                y=0.95, x=0.02,
                xanchor="left", yanchor="top",
                pad=dict(t=0, r=10),
                bgcolor='rgba(30, 41, 59, 0.85)',
                bordercolor='rgba(255, 255, 255, 0.2)',
                font=dict(color='#ffffff', size=13),
                buttons=[
                    dict(
                        label="▶ Tự Động Quay 360°",
                        method="animate",
                        args=[None, dict(
                            frame=dict(duration=50, redraw=False),
                            fromcurrent=True,
                            mode="immediate",
                            loop=True
                        )]
                    ),
                    dict(
                        label="⏸ Tạm Dừng (Tự Xoay Chuột 360°)",
                        method="animate",
                        args=[[None], dict(
                            frame=dict(duration=0, redraw=False),
                            mode="immediate",
                            transition=dict(duration=0)
                        )]
                    )
                ]
            )
        ]
    )

    print("✨ Đã dựng thành công Khung Mây Điểm 3D Nền Đen & Quỹ Đạo Camera!")
    fig.show()

# ── Đọc File Mây Điểm & File Camera Poses trên Máy Local ─────────────────────────
local_base = os.path.join(os.getcwd(), "counter_demo_video")
ply_path = os.path.join(local_base, "step1_initial_points", "initial_point_cloud.ply")
poses_path = os.path.join(local_base, "poses_bounds.npy")
fixed_cams_path = os.path.join(local_base, "fixed_cameras.json")

if not os.path.exists(ply_path):
    ply_path = r"C:\Users\YUT9HC\Desktop\Z\thesis-all\counter_demo_video\step1_initial_points\initial_point_cloud.ply"
    poses_path = r"C:\Users\YUT9HC\Desktop\Z\thesis-all\counter_demo_video\poses_bounds.npy"
    fixed_cams_path = r"C:\Users\YUT9HC\Desktop\Z\thesis-all\counter_demo_video\fixed_cameras.json"

if os.path.exists(ply_path):
    display_interactive_point_cloud_plotly(ply_path, poses_path, fixed_cams_path)
else:
    print(f"⚠️ Không tìm thấy file tại {ply_path}. Vui lòng kiểm tra lại thư mục counter_demo_video!")